 %%
 ### <font color="orange">🧠 GIAI ĐOẠN 3: KỶ NGUYÊN LLMs</font>
 #### Bài 25: Tự code cơ chế "Attention Is All You Need" từ giấy trắng

 **1. Sự sụp đổ của Seq2Seq (Bài 25)**
 Ở bài 25, em nén một câu tiếng Anh dài 1000 chữ vào đúng một cục `Context Vector` bé xíu rồi bắt Decoder dịch. Chắc chắn nó sẽ quên các từ ở đầu câu (gọi là Nút thắt cổ chai - Bottleneck).

 **2. Giải pháp: Attention (Sự Chú Ý)**
 Năm 2017, Google tung ra bài báo "Attention Is All You Need".
 Thay vì nén tất cả vào 1 cục, khi Decoder chuẩn bị dịch từ "Tôi", nó sẽ **liếc mắt nhìn lại toàn bộ 1000 chữ tiếng Anh ban đầu**. Nó sẽ tính toán xem: *"Với chữ 'Tôi' này, mình nên 'chú ý' (cho điểm cao) vào chữ tiếng Anh nào nhất?"*.

 **3. Khái niệm Q, K, V (Query, Key, Value)**
 Hãy tưởng tượng em đang tìm video trên YouTube:
 * **Query (Q - Câu truy vấn)**: Thứ em đang tìm (Ví dụ Decoder đang cần tìm danh từ để dịch).
 * **Key (K - Tiêu đề video)**: Tiêu đề của tất cả các từ bên Encoder để em đối chiếu xem có khớp với Q không.
 * **Value (V - Nội dung video)**: Ý nghĩa thực sự của từ đó sẽ được lấy ra nếu Q khớp với K.

 **4. Công thức Toán học cốt lõi của thế kỷ 21 (Scaled Dot-Product Attention):**
 Để tính ra sự chú ý, máy tính làm 4 bước:
 > $ \color{orange} Attention(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V $

 * **Bước 1**: Lấy $Q$ nhân ma trận với $K$ chuyển vị ($K^T$) để chấm điểm độ "khớp" nhau giữa Query và các Key.
 * **Bước 2**: Chia điểm đó cho căn bậc 2 của kích thước vector ($\sqrt{d_k}$) để số không bị quá to (Scale).
 * **Bước 3**: Đưa qua hàm $\text{softmax}$ để biến các điểm số thành phần trăm (tổng = 100%).
 * **Bước 4**: Lấy phần trăm đó nhân ma trận với $V$ để trích xuất ra thông tin quan trọng nhất.

 %%

In [3]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V):
    """
    Hàm tính toán Cơ chế Sự chú ý (Attention) cốt lõi của Transformer.
    Input:
      Q (Query): Kích thước (batch_size, số_từ_đang_dịch, d_k)
      K (Key)  : Kích thước (batch_size, tổng_số_từ_đã_đọc, d_k)
      V (Value): Kích thước (batch_size, tổng_số_từ_đã_đọc, d_v)
    """
    
    # Lấy d_k là chiều cuối cùng của ma trận Q
    d_k = Q.size(-1)
    
    # ====================================================================
    # KHU VỰC GIẤY TRẮNG CỦA EM
    # ====================================================================
    
    # 1. Tính Scores = Q nhân ma trận với K chuyển vị.
    # LƯU Ý KÍCH THƯỚC: Em chỉ được phép chuyển vị 2 chiều cuối của K.
    # PyTorch: K.transpose(-2, -1)
    # Nhân ma trận trong PyTorch: torch.matmul(A, B) hoặc A @ B
    scores = Q @ K.transpose(-2, -1)
    
    # 2. Scale (Chia Scores cho căn bậc 2 của d_k)
    # PyTorch/Python: math.sqrt(số_nào_đó)
    scaled_scores = scores / (math.sqrt(d_k))
    
    # 3. Attention Weights = Tính Softmax của scaled_scores
    # PyTorch: torch.softmax(tensor, dim=-1)
    attention_weights = torch.softmax(scaled_scores, dim=-1)
    
    # 4. Output = Attention Weights nhân ma trận với V
    output = attention_weights @ V
    
    return output, attention_weights

# %%
# ====================================================================
# KHU VỰC KIỂM THỬ (UNIT TESTS) - KHÔNG ĐƯỢC SỬA BÊN DƯỚI NÀY
# ====================================================================
def main():
    print("--- THỬ THÁCH CODE 'ATTENTION IS ALL YOU NEED' ---")
    
    # Cố định seed để ra đúng 1 kết quả
    torch.manual_seed(42)
    
    batch_size = 1
    seq_len_q = 1   # Đang xét 1 từ đang dịch (Ví dụ chữ "Tôi")
    seq_len_k = 4   # Nhìn lại 4 từ tiếng Anh ở Encoder ("I", "am", "a", "student")
    d_k = 3         # Vector 3 chiều
    
    # Tạo các tensor ngẫu nhiên đóng vai trò Q, K, V
    Q = torch.rand(batch_size, seq_len_q, d_k)
    K = torch.rand(batch_size, seq_len_k, d_k)
    V = torch.rand(batch_size, seq_len_k, d_k)
    
    print("\nĐang tính toán bằng hàm của em...")
    try:
        my_output, my_weights = scaled_dot_product_attention(Q, K, V)
        
        # Đối chiếu với hàm Attention chuẩn đã được tối ưu của PyTorch bằng C++
        pytorch_output = F.scaled_dot_product_attention(Q, K, V)
        
        print("\nTrọng số Attention (Mức độ chú ý của chữ 'Tôi' vào 4 chữ kia):")
        print(my_weights.detach().numpy())
        
        # Kiểm tra tính chính xác
        if torch.allclose(my_output, pytorch_output, atol=1e-5):
            print("\n✅ KIỂM ĐỊNH THÀNH CÔNG: CHÍNH XÁC 100%!")
            print("Chúc mừng! Em vừa tự code thành công lõi toán học của ChatGPT!")
        else:
            print("\n❌ KẾT QUẢ SAI LỆCH VỚI PYTORCH. Hãy xem lại công thức!")
            
    except Exception as e:
        print(f"\n❌ Lỗi Code: {e}")
        print("💡 Đừng hoảng! Lỗi Matrix Dimension (kích thước ma trận) là đặc sản của AI Engineer. Hãy kiểm tra lại hàm transpose và matmul.")

if __name__ == "__main__":
    main()

--- THỬ THÁCH CODE 'ATTENTION IS ALL YOU NEED' ---

Đang tính toán bằng hàm của em...

Trọng số Attention (Mức độ chú ý của chữ 'Tôi' vào 4 chữ kia):
[[[0.2571795  0.23983318 0.2247103  0.27827707]]]

✅ KIỂM ĐỊNH THÀNH CÔNG: CHÍNH XÁC 100%!
Chúc mừng! Em vừa tự code thành công lõi toán học của ChatGPT!
